In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# AlexNet and Learned Image Representations

The LeNet of that section learns everything it computes from examples,
its convolution kernels as well as its classifier. LeCun and his colleagues
trained the first such network in 1989, yet for the next two decades most
vision systems worked differently. They separated representation from
prediction: hand-designed operators such as SIFT [@Lowe.2004], SURF
[@Bay.Tuytelaars.Van-Gool.2006], HOG [@Dalal.Triggs.2005], and bags
of visual words [@Sivic.Zisserman.2003] turned pixels into feature
vectors, and a linear or kernel method learned a classifier on top. The
training loss could improve the classifier but never the features, so
progress depended on people inventing better features. We call a
representation learned when the loss decides what the features compute as
well. Why did learned features take so long to beat engineered ones, and
what changed when they did?

The change came at the ImageNet challenge of 2012, which asked a system to
sort photographs into 1000 categories and scored it by its *top-5 error*, the
fraction of validation images whose correct label is not among the system's five
best guesses. AlexNet [@Krizhevsky.Sutskever.Hinton.2012] replaced the
two-stage pipeline with one deep convolutional network trained end to end,
and reached a top-5 error of 15.3%, against 26.2% for the best entry built on
engineered features. The year before, the winning error had fallen by about
two points. This time it fell by more than ten.

AlexNet's architecture is the template of LeNet scaled up: convolutional
stages that shrink the resolution and grow the number of channels, followed
by fully connected layers. The template was more than twenty years old,
which only sharpens the question. Three things made the scale-up work in
2012 when it had not before: a labeled dataset large enough for tens of
millions of parameters, GPUs fast enough to train them in days, and
techniques (ReLU activations, dropout, data augmentation) that kept a
network of this size trainable and kept it from overfitting.

This section explains why each of the three mattered, builds AlexNet, counts
where its parameters and its arithmetic sit, and trains it on Fashion-MNIST.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

## Learned Features, Data, and Hardware

Learning the features was not a new idea in 2012. Besides LeNet, sparse
coding had learned image filters without any labels
[@olshausen1996emergence]. What was new is that the learned features
won on a large benchmark, and AlexNet's first layer shows what the network
chose to compute. the figure displays its 96 learned kernels of
$11 \times 11$ pixels: oriented edges at several frequencies and patches of
color, close to the oriented-gradient operators that SIFT and HOG compute by
design. Nobody told the network to look for edges. It found these operators
because they reduce the classification loss, and it found the operators of
its higher layers the same way, where no designer had a good guess to offer.

![The 96 kernels of the first convolutional layer of AlexNet, learned on ImageNet. Reproduction courtesy of @Krizhevsky.Sutskever.Hinton.2012.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/filters.png)

### Data

A model with tens of millions of parameters, fitted to the ten thousand
images of a benchmark such as PASCAL VOC, simply memorizes them. The
ImageNet dataset [@Deng.Dong.Socher.ea.2009] changed the scale: its
2012 challenge [@russakovsky2015imagenet] supplied about 1.2 million
labeled training images in 1000 categories. Even that leaves some fifty
parameters for every training image, so AlexNet's authors stretched the
data further. They trained on random $224 \times 224$ crops of images
rescaled to $256 \times 256$, and on their mirror images. A crop can start
at any of about $32 \times 32$ offsets and can be mirrored or not, so each
image yields about $32 \cdot 32 \cdot 2 = 2{,}048$ different inputs. These are
overlapping views of one photograph, of course, and even with them the
network overfit without dropout [@Krizhevsky.Sutskever.Hinton.2012].

### Hardware

Training a deep network repeats the same small computations at enormous
scale. A convolution applies one kernel at every output position and every
channel, and each of these products is independent of the others. A
*graphics processing unit* (GPU) is built for such work: instead of a few
fast, complicated cores it has hundreds or thousands of simple ones and a
much wider memory bus. Each of the two NVIDIA GTX 580s on which AlexNet was
trained performs about 1.6 trillion floating-point operations per second
(TFLOPS) in single precision and reads 192 GB of memory per second; a
quad-core desktop CPU of 2011 peaked at about 0.2 TFLOPS and 20 GB/s.

A short calculation shows why this mattered. A network of AlexNet's size
needs about a billion multiply-adds to classify one $224 \times 224$ color
image. A training step costs about three times as many: one pass forward
and, on the way back, two products of the same size in each layer, one for
the gradient of its weights and one for the gradient handed to the layer
below. With two floating-point operations per multiply-add, the 90 passes
over 1.2 million images used in the paper take about $10^9 \cdot 3 \cdot 2
\cdot 1.2 \times 10^6 \cdot 90 \approx 6.5 \times 10^{17}$ operations. The
two GPUs need two to three days for that at their peak, and the actual run
took five to six days. A CPU at its peak would have needed more than a
month, for every attempt. Alex Krizhevsky wrote the GPU code for the
convolutions himself
([cuda-convnet](https://code.google.com/archive/p/cuda-convnet/)); today the
same network takes a dozen lines, as below.

## The AlexNet Architecture

the figure places AlexNet beside the LeNet of
that section, both in the versions that this book trains on
Fashion-MNIST. Both consist of convolutional stages followed by a fully
connected head. AlexNet changes the size of every part and three of its
components: the activation function, the pooling, and the regularizer.

![LeNet as built in that section (left) and AlexNet as built in this section (right), both for single-channel Fashion-MNIST images and ten classes; the input is at the bottom. AlexNet is deeper and wider, uses ReLU activations and max-pooling, and regularizes its dense layers with dropout.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/arch-alexnet.svg)

### Depth and Width

AlexNet has eight layers with weights: five convolutional layers and three
fully connected layers, against LeNet's two and three. Its images are eight
times wider and taller than Fashion-MNIST's, so the first layer uses an
$11 \times 11$ kernel with stride 4 to cover a large patch of the image at
once and to reduce the resolution early. The second layer uses
$5 \times 5$ kernels and the last three $3 \times 3$ kernels. Max-pooling
with $3 \times 3$ windows and stride 2 follows the first, second, and fifth
convolutional layers. It replaces LeNet's average pooling, because a
maximum keeps a strong response that an average would dilute
(that section), and its windows overlap. The widths grow
accordingly: 96 to 384 channels where LeNet has 6 and 16, and 4096 hidden
units where LeNet has 120 and 84.

Our version leaves out two features of the paper's network, and neither is
missed. The paper split every layer between two GPUs, because one GPU with
3 GB of memory could not hold the model; a single GPU now can. It also
normalized each activation by the activations of neighboring channels, a
step that later networks dropped.

### ReLU Activations

LeNet uses saturating activations: the sigmoid in our version, a scaled
$\tanh$ in the original of 1998. Each sigmoid that the gradient passes on
its way back to the first layer multiplies it by at most $1/4$, which is one
reason why LeNet needed a learning rate as large as 2.0
(that section). AlexNet uses the ReLU,
$\textrm{ReLU}(x) = \max(x, 0)$ (that section),
whose derivative is 1 for every positive input. A ReLU, unlike a sigmoid,
therefore does not shrink the gradient on its active units. What remains is
the scaling by the weights of each layer, which the initialization below is
chosen to control. The ReLU is also cheaper to compute, since it needs no
exponential. The difference is not subtle: on a small convolutional network
for CIFAR-10, @Krizhevsky.Sutskever.Hinton.2012 reached a training
error of 25% six times faster with ReLUs than with $\tanh$.

### Dropout and Augmentation

The two fully connected layers of 4096 units hold most of the parameters and
are where the model can memorize the training set. AlexNet applies dropout
with probability 0.5 to their outputs (that section), whereas LeNet
has no explicit regularizer at all. The crops and mirror images described
above, together with random changes of color, add variation to the input.
Augmentation has since grown into a large part of how convolutional
networks are trained (that section).

### Implementation

The code mirrors the figure from bottom to top. We initialize the
weights explicitly instead of relying on the library's defaults. LeNet's
Xavier initialization keeps the variance of the activations constant through
linear layers, but a ReLU sets about half of its inputs to zero and halves
the second moment of what passes through it. *He initialization*
(that section) compensates by doubling the variance. Every
weight of a layer with $n_\textrm{in}$ inputs per output is drawn from a
normal distribution with mean zero and variance $2/n_\textrm{in}$, where
$n_\textrm{in} = c_\textrm{i} k^2$ for a convolution. All biases start at
zero, which is the default of every library here except PyTorch.

NNX layers draw their weights in the constructor, so every layer receives the
initializer.

In [ ]:
class AlexNet(d2l.Classifier):
    def __init__(self, lr=0.1, num_steps=4690, num_classes=10,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = (nnx.Rngs(params=d2l.get_key(), dropout=d2l.get_key())
                if rngs is None else rngs)
        he = nnx.initializers.variance_scaling(2.0, 'fan_in', 'normal')
        init = dict(kernel_init=he, rngs=rngs)

        def max_pool(x):  # 3x3 windows, stride 2
            return nnx.max_pool(x, window_shape=(3, 3), strides=(2, 2))

        def flatten(x):  # One vector per example
            return x.reshape((x.shape[0], -1))

        self.net = nnx.Sequential(
            nnx.Conv(1, 96, kernel_size=(11, 11), strides=4, padding='VALID',
                     **init), nnx.relu, max_pool,
            nnx.Conv(96, 256, kernel_size=(5, 5), **init), nnx.relu,
            max_pool,
            nnx.Conv(256, 384, kernel_size=(3, 3), **init), nnx.relu,
            nnx.Conv(384, 384, kernel_size=(3, 3), **init), nnx.relu,
            nnx.Conv(384, 256, kernel_size=(3, 3), **init), nnx.relu,
            max_pool,
            flatten,  # 5 x 5 x 256 values per image
            nnx.Linear(5 * 5 * 256, 4096, **init), nnx.relu,
            nnx.Dropout(0.5, rngs=rngs),
            nnx.Linear(4096, 4096, **init), nnx.relu,
            nnx.Dropout(0.5, rngs=rngs),
            nnx.Linear(4096, num_classes, **init))

To check the code against the figure, we pass a single-channel
$224 \times 224$ image through the network and print the output shape of each
layer.

In [ ]:
AlexNet().layer_summary((1, 224, 224, 1))

The shapes follow the rules of that section. The first convolution
maps 224 to $\lfloor (224 - 11)/4 \rfloor + 1 = 54$, and each $3 \times 3$
pooling with stride 2 maps $n$ to $\lfloor (n - 3)/2 \rfloor + 1$: 54 to 26,
26 to 12, and 12 to 5. The padded convolutions in between keep the
resolution, so the last convolutional stage hands $256 \times 5 \times 5 =
6400$ numbers to the first fully connected layer.

### Parameters and Multiply-Adds

The shapes determine the size and the cost of every layer, counted as for
LeNet (that section): a convolutional layer with $c_\textrm{i}$ input
channels, $c_\textrm{o}$ output channels, and a $k \times k$ kernel applies
each of its $c_\textrm{i} c_\textrm{o} k^2$ weights once at every output
position, and a fully connected layer uses each weight once per example.
the table applies these counts to AlexNet.

<!-- tbl-caption:Parameters and multiply-adds per example of the layers of AlexNet, for one single-channel $224 \times 224$ image and ten classes. Pooling, activation, and dropout layers have no parameters. -->

| layer | output | parameters | multiply-adds |
|:--|:--|--:|--:|
| $11\times11$ convolution, 1 to 96, stride 4 | 96@54×54 | $11{,}712$ | $33.9$M |
| $5\times5$ convolution, 96 to 256 | 256@26×26 | $614{,}656$ | $415.3$M |
| $3\times3$ convolution, 256 to 384 | 384@12×12 | $885{,}120$ | $127.4$M |
| $3\times3$ convolution, 384 to 384 | 384@12×12 | $1{,}327{,}488$ | $191.1$M |
| $3\times3$ convolution, 384 to 256 | 256@12×12 | $884{,}992$ | $127.4$M |
| fully connected, 6400 to 4096 | 4096 | $26{,}218{,}496$ | $26.2$M |
| fully connected, 4096 to 4096 | 4096 | $16{,}781{,}312$ | $16.8$M |
| fully connected, 4096 to 10 | 10 | $40{,}970$ | $0.04$M |
| total | | $46{,}764{,}746$ | $938.1$M |

The two kinds of layers divide the work as in LeNet, only more sharply. The
convolutional layers hold 8% of the 47 million parameters and perform 95% of
the multiply-adds. The fully connected layers hold the other 92% of the
parameters, about 170 MB in 32-bit floating point, and perform 5% of the
arithmetic. Against LeNet's 61,706 parameters and 416,520 multiply-adds per
image, AlexNet has about 760 times the parameters and performs about 2,250
times the arithmetic. The network of the paper, with three input channels and 1000 classes, has
about 60 million parameters, most of them in the same fully connected layers.

## Training on Fashion-MNIST

Training AlexNet on ImageNet takes days even on a modern GPU, so we train it
on Fashion-MNIST, as we did LeNet. Its $28 \times 28$ images are too small
for an $11 \times 11$ kernel with stride 4, so we upsample them to
$224 \times 224$ with the `resize` argument of `d2l.FashionMNIST`.
Upsampling multiplies the number of pixels by 64 without adding
information. We accept the waste in order to train the architecture as it
is.

We train with minibatch SGD for 10 epochs, a third of LeNet's 30, with
minibatches of 128 images. With ReLUs in place of sigmoids, a learning rate
far below LeNet's 2.0 suffices.

### Learning-Rate Schedule

A constant learning rate suits neither the first steps nor the last. At the
start the weights are random, and a full-size step can throw the network
into a region from which it recovers only slowly. At the end, a large rate
keeps the weights jumping around a good solution instead of settling into
it. AlexNet's authors took care of the end by hand: they divided the rate by
10 whenever the validation error stopped improving. We use a schedule that handles both ends without
supervision. It is a later default and not part of AlexNet; the following
sections use it too, so that their results can be compared. A linear
*warmup* raises the rate from zero to its peak $\eta_0$ over the first
$t_\textrm{w}$ steps, and a half-cosine then lowers it to zero at the last
step $T$:

$$
\eta_t = \frac{\eta_0}{2} \left(1 + \cos \frac{\pi (t - t_{\textrm{w}})}{T - t_{\textrm{w}}}\right)
\quad \textrm{for } t \geq t_{\textrm{w}}.
$$

At $t = t_\textrm{w}$ the cosine is 1 and the rate is $\eta_0$; halfway
through the decay it is $\eta_0/2$; at $t = T$ it is zero. Since the first
steps are small, the peak can be higher than a rate that would be safe from
the first step on. We train AlexNet at a peak of 0.03. How high the peak can
go depends on the network, and that section gives each of its
networks the largest rate at which it trains reliably. The function
`cosine_warmup` returns the factor by which the schedule multiplies the peak
rate at step $t$; we save it in the `d2l` package for later sections.
that section compares learning-rate schedules in depth.

In [ ]:
def cosine_warmup(t, num_steps, warmup_steps):
    """Learning-rate factor: linear warmup from 0, then a half-cosine to 0."""
    t = jnp.asarray(t, jnp.float32)  # Arguments may be traced arrays
    warmup = jnp.where(t < warmup_steps, t / warmup_steps, 1.0)
    progress = jnp.clip((t - warmup_steps) / (num_steps - warmup_steps), 0, 1)
    return warmup * 0.5 * (1 + jnp.cos(jnp.pi * progress))

`AlexNet` steps the schedule after every minibatch. Its argument
`num_steps` is $T$, the number of minibatches in the whole run, and the
warmup covers the first tenth of them: one epoch of the ten.

In [ ]:
@d2l.add_to_class(AlexNet)
def configure_optimizers(self):
    return optax.sgd(lambda t: self.lr * cosine_warmup(
        t, self.num_steps, self.num_steps // 10))

### Comparison with LeNet

What should we expect? Fashion-MNIST has neither the number of images nor
the detail that AlexNet's capacity is meant for. The run checks the
implementation and measures the cost. It cannot show what made AlexNet win
on ImageNet, and the gain over LeNet should be small. Before training we
keep a copy of the 96 kernels of the first layer, so that we can see
afterwards what training changed.

In [ ]:
def first_kernels(model):
    """The 96 kernels of 11 x 11 weights of the first layer, as NumPy."""
    kernel = d2l.numpy(model.net.layers[0].kernel[...])  # (11, 11, 1, 96)
    return kernel[:, :, 0].transpose(2, 0, 1)

data = d2l.FashionMNIST(batch_size=128, resize=(224, 224))
model = AlexNet(lr=0.03, num_steps=10 * len(data.train_dataloader()))
K0 = first_kernels(model)

In [ ]:
trainer = d2l.Trainer(max_epochs=10, num_gpus=1)
trainer.fit(model, data)

AlexNet reaches a validation accuracy of about 91% after 10 epochs, with a
training loss of about 0.22 and a validation loss of about 0.25. Over the last
epochs, as the learning rate approaches zero, the accuracy changes by less
than a point from one epoch to the next, so the last epoch is a fair summary
of the run. LeNet reaches about 88–90% after 30 epochs. The larger network thus passes
LeNet's accuracy in a third of the epochs, but only by a point or two, and
it performs 2,250 times the arithmetic per image to do so. Nor can we credit
its size alone with even this gain, since the two runs differ in activation,
initialization, and schedule as well.

### The Learned First Layer

LeNet's six kernels of $5 \times 5$ weights were too small to tell the eye
much (that section). AlexNet's 96 kernels of $11 \times 11$ weights
are large enough to read as pictures, and the figure showed what
they become on ImageNet. What did ours learn from Fashion-MNIST? We subtract
the copy taken before training from the kernels after it and draw the
difference, each kernel on its own gray scale.

In [ ]:
change = first_kernels(model) - K0
d2l.show_images(change, 6, 16, scale=0.6, cmap='gray')
print(f'mean size of the change: {np.abs(change).mean() / np.abs(K0).mean():.3f}'
      ' of the mean size of the initial weights')

Ten epochs moved the kernels by only about 4% of the size of their random
initial values, so a trained kernel still looks like the noise it started
from. The change itself is not noise. It is smooth: brightness ramps and
broad bars at various orientations, the low-frequency relatives of the edge
detectors of the figure. The sharp edges and the color blobs of
that figure are missing, and the input explains both. Our images were
upsampled eightfold, so they contain no detail finer than about eight
pixels, which an $11 \times 11$ kernel can only see as a gradual change. And
they have a single channel. The first layer learned what the data contain.

Most of AlexNet's capacity, as the table showed, sits in a
fully connected head that does little of the arithmetic and needs dropout to
keep it from memorizing the training set. The next section asks whether a
network needs that head at all.

## Summary

Learned features overtook hand-designed ones when three things were in
place at once: a dataset large enough to constrain tens of millions of
parameters, GPUs that could compute their gradients in days, and ReLU
activations, dropout, and data augmentation to keep such a network
trainable. With them AlexNet, LeNet's template made deeper and wider, cut
the ImageNet top-5 error from 26% to 15%.
Counting its layers shows that the convolutions perform 95% of the
arithmetic while the fully connected head holds 92% of the parameters. On
Fashion-MNIST, which cannot use that capacity, AlexNet passes LeNet's
accuracy by a point or two in a third of the epochs, at 2,250 times the
arithmetic per image.

## Exercises

1. **Memory and compute ledger.** Extend the table to the
   memory that `AlexNet` needs in 32-bit floating point.
    1. Compute the number of activations that the forward pass of one
       $224 \times 224$ image stores, layer by layer, counting the output
       of every convolutional, pooling, and fully connected layer. Where is
       the memory for activations concentrated, and where is the memory for
       parameters concentrated?
    1. During training, the parameters, their gradients, and the stored
       activations of a whole minibatch must fit in memory. At which batch
       size does the memory for activations exceed the memory for
       parameters and gradients?
    1. Repeat the parameter count of the table for the
       ImageNet version, with three input channels, 1000 classes, and
       $227 \times 227$ inputs, which make the last convolutional stage
       $6 \times 6$ instead of $5 \times 5$. Compare your total with the 60
       million parameters reported by
       @Krizhevsky.Sutskever.Hinton.2012 .

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 10.16.*

1. **Receptive fields of the early layers.** A stack of layers with kernel
   sizes $k_i$ and strides $s_i$ gives its top unit a receptive field of
   side $r = 1 + \sum_{i} (k_i - 1) \prod_{j<i} s_j$
   (that section). Apply it to `AlexNet`, counting each
   max-pooling layer as a layer with its own kernel size and stride, to
   compute the receptive field of one unit after each of the five
   convolutional layers. What fraction of the $224 \times 224$ input side
   does each receptive field span, and where does a unit first see the
   whole image?

    *Adapted from Simon Prince,
    [Understanding Deep Learning](https://udlbook.github.io/udlbook/),
    Problem 10.17.*

1. [code] **Change per layer.** Training moved the first-layer kernels by
   about 4% of the size of their initial values.
    1. Keep a copy of every weight array before training and compute the
       same ratio for each of the eight layers with weights. Which layers
       moved most, and which least? Relate the order to the number of
       weights in each layer.
    1. The text attributes the smoothness of the first layer's change to
       the eightfold upsampling. Test this: train on images resized to
       $112 \times 112$, which are upsampled fourfold (the last
       convolutional stage is then $2 \times 2$, so the first fully
       connected layer has 1024 inputs), and draw the change of the first
       layer again. At which scale does it vary now?

1. [code] **Dropout.** The fully connected layers of `AlexNet` hold 92% of
   its parameters, and dropout is its only regularizer.
    1. Train `AlexNet` as in this section with both dropout layers removed,
       and compare the training loss, the validation loss, and the
       validation accuracy after 10 epochs with those of the original.
    1. Train both networks on a random subset of 6,000 training images for
       the same number of gradient steps. How do the differences change, and
       why?

1. **Arithmetic intensity.** ● The *arithmetic intensity* of a layer is
   the number of multiply-adds it performs per byte of parameters it reads
   from memory. Use the table.
    1. Compute the arithmetic intensity of every layer of `AlexNet`, once
       for a batch of 1 image and once for a batch of 128 images, assuming
       that each parameter is read once per batch.
    1. Consider a chip that performs 10 multiply-adds in the time it reads
       one byte from memory. Assume that each layer takes the longer of its
       arithmetic time and its parameter-reading time. A designer can spend
       extra chip area on doubling either the arithmetic throughput or the
       memory bandwidth. Which choice speeds up single-image inference more,
       and which speeds up training at batch size 128 more?

1. [code] **Batch size, throughput, and memory.** ● Train `AlexNet` as in
   this section while varying only the batch size.
    1. Using your answer to problem
       1, predict the largest
       training batch size that fits in the memory of your GPU.
    1. Starting from a batch size of 32 and doubling it, find the largest
       batch size that trains without running out of memory. Compare it
       with your prediction and name the main source of any discrepancy.
    1. Measure the throughput in images per second over a fixed number of
       training steps for at least four batch sizes up to that limit.
       Beyond which batch size does throughput stop growing, and why?

[Discussions](https://d2l.discourse.group/t/18001)